# Human Activity Recognition with Wifi

In [ ]:
import numpy as np
import numpy.random as rnd
import torch
import matplotlib.pyplot as plt

from models.DopplerDataset import * 
from models.Architectures import *
from utilities.training import *
from utilities.plotting import *
from utilities.data_processing import *

from time import time 
from torchvision.transforms import Compose, ToTensor, RandomHorizontalFlip, RandomVerticalFlip
from torch.utils.data import DataLoader
from os import listdir, makedirs, path
from glob import glob
from tqdm import tqdm

%load_ext autoreload
%autoreload 2

## Separating training dataset and test dataset

In [ ]:
DEBUG_MODE                 = True

TRAIN_DATASET_PATH  = "doppler_traces/S1*"
TEST_DATASET_PATH   = "doppler_traces/S2*"
DOPPLER_TRACE_SIZE  = 340

LABELS     = ['W', 'E', 'R', 'J', 'L']
ACTIVITIES = ['Walking', 'Empty', 'Running', 'Jumping', 'Sitting']
labels_map = {label:idx for idx, label in enumerate(LABELS)}

create_train_dataset(TRAIN_DATASET_PATH, DOPPLER_TRACE_SIZE, LABELS)
create_test_dataset(TEST_DATASET_PATH, DOPPLER_TRACE_SIZE, LABELS)

## Training of SHARP architecure (no batch normalization)

In [ ]:
batch_size = 128
train_dataset  = DopplerDataset("doppler_traces_train", labels_map, db_conversion=True, normalization=False)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
train_dataset.getInfo()

test_dataset  = DopplerDataset("doppler_traces_test", labels_map, db_conversion=True, normalization=False)
test_dataloader = DataLoader(test_dataset, batch_size=4, shuffle=False)
test_dataset.getInfo()

print("Training dataset length: ", len(train_dataset))
print("Test dataset length: ", len(test_dataset))

plot_dataset(3, 3, train_dataset, ACTIVITIES, LABELS)

In [ ]:
sharp_model = SHARP(n_features=len(ACTIVITIES), batchNorm=False)

learning_rate = 1e-4
device    = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
loss_fn   = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=sharp_model.parameters(), lr=learning_rate)

if DEBUG_MODE:
    #### CHECKING DATA TYPES 
    x, y = next(iter(train_dataloader))
    print(x.dtype)
    print(next(sharp_model.parameters()).dtype)
    ### DEVICE
    print(f"Device: {device}")

In [ ]:
epochs = 20
train_loss, test_loss, train_acc, test_acc, _ = train_model(sharp_model, train_dataloader, test_dataloader, epochs, loss_fn, optimizer, device, verbosity=True)

In [ ]:
plot_loss(train_loss, test_loss, train_acc, test_acc, "SHARP model, no batch normalization")

### Training of SHARP (with BatchNormalization)

In [ ]:
train_dataset  = DopplerDataset("doppler_traces_train", labels_map, db_conversion=False, normalization=True)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
train_dataset.getInfo()

test_dataset  = DopplerDataset("doppler_traces_test", labels_map, db_conversion=False, normalization=True)
test_dataloader = DataLoader(test_dataset, batch_size=4, shuffle=False)
test_dataset.getInfo()

plot_dataset(3, 3, train_dataset, ACTIVITIES, LABELS)

In [ ]:
sharp_model = SHARP(n_features=len(ACTIVITIES), batchNorm=True)

learning_rate = 5e-5
device    = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
loss_fn   = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=sharp_model.parameters(), lr=learning_rate, weight_decay=1e-5)
sharp_model

In [ ]:
epochs = 20
train_loss, test_loss, train_acc, test_acc, _ = train_model(sharp_model, train_dataloader, test_dataloader, epochs, loss_fn, optimizer, device, verbosity=True)

In [ ]:
plot_loss(train_loss, test_loss, train_acc, test_acc, "SHARP model, with batch normalization")

## Metrics

In [ ]:
# maybe add a function to save the best model based on loss ? 

metrics = compute_metrics("all", sharp_model, test_dataset, LABELS, device, True)
save_best_F1_model(sharp_model, metrics) # based on F1-score

In [ ]:
plot_confusion_matrix(metrics["cm"], LABELS)

In [ ]:
plot_f1_score(metrics["precision"], metrics["recall"], metrics["f1"], LABELS)

### Save best model

In [ ]:
save_best_model(sharp_model, metrics)